# SQuAD Chunking Strategy Comparison — Coreference Metrics

Compares coreference-aware chunking quality metrics across all 4 chunking strategies using SQuAD Wikipedia articles.

**Strategies:**
- `baseline_258_tok` — fixed 258-token windows (tokenized/decoded)
- `most_recent_low_acc_258t_w128_wtd` — discourse-aware, window=128, weighted
- `most_recent_258t_w254` — discourse-aware, window=254, unweighted
- `nonlinear_258t_w254` — nonlinear discourse, window=254, unweighted

**Metrics:**
- `cluster_break_rate` — fraction of coreference chains split across multiple chunks (lower is better)
- `edge_cut_rate` — fraction of consecutive mention-pairs crossing a chunk boundary (lower is better)
- `entity_concentration` — avg per-chunk dominance of the most-mentioned entity (higher is better)

**Cluster source:** `clusters/squad/<slug>.json` (generated by `generate_squad_clusters.py` via f-coref)

> **Note on baseline:** Baseline chunks are tokenized/decoded (lowercased, spaces around punctuation),
> so f-coref character offsets from the original article text do not align exactly.
> Baseline coreference metrics are **approximate** (proportional boundary assignment).

In [ ]:
import json
import re
import bisect
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(Path('.').resolve()))
from metrics import cluster_break_rate, edge_cut_rate, entity_concentration

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.titlesize': 14,
})

PARQUET_PATH = Path('squad_testing/data/squad_wiki_full_articles.parquet')
CHUNKS_BASE  = Path('squad_testing/chunks')
CLUSTERS_DIR = Path('clusters/squad')

STRATEGIES = [
    'baseline_258_tok',
    'most_recent_low_acc_258t_w128_wtd',
    'most_recent_258t_w254',
    'nonlinear_258t_w254',
]
STRATEGY_SHORT = {
    'baseline_258_tok':                  'baseline',
    'most_recent_low_acc_258t_w128_wtd': 'mr_low_acc',
    'most_recent_258t_w254':             'mr_w254',
    'nonlinear_258t_w254':               'nonlinear',
}
COLORS = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

In [ ]:
def slugify(title: str) -> str:
    return re.sub(r'[\\/*?:"<>|]', '-', title).strip()


def load_chunks(strategy: str, title: str) -> list[str]:
    """Load chunk strings for a strategy and article title (with spaces)."""
    if strategy == 'baseline_258_tok':
        slug = slugify(title)
        path = CHUNKS_BASE / strategy / slug / 'chunks_output.json'
    else:
        fname = title.replace(' ', '_') + '.json'
        path = CHUNKS_BASE / strategy / fname
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)['chunks']


def load_clusters(title: str) -> list:
    """Load f-coref clusters for an article. Returns [] if file missing."""
    slug = slugify(title)
    path = CLUSTERS_DIR / f'{slug}.json'
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)


def chunk_starts_nonbaseline(chunks: list[str]) -> list[int]:
    """Build sorted chunk-start character positions for non-baseline strategies.

    Non-baseline chunks are direct character slices of the article text, so
    boundaries are simply cumulative lengths.
    """
    starts = []
    pos = 0
    for chunk in chunks:
        starts.append(pos)
        pos += len(chunk)
    starts.append(pos)  # sentinel
    return starts


def chunk_starts_baseline(article_len: int, n_chunks: int) -> list[int]:
    """Approximate proportional boundaries for baseline chunks.

    Baseline chunks are tokenized/decoded so their text differs from the
    original. We divide the original article evenly to approximate placement.
    """
    if n_chunks == 0:
        return [0, article_len]
    step = article_len / n_chunks
    starts = [int(i * step) for i in range(n_chunks)]
    starts.append(article_len)
    return starts


def mention_to_chunk_id(m_start: int, starts: list[int]) -> int:
    idx = bisect.bisect_right(starts, m_start) - 1
    return max(0, min(idx, len(starts) - 2))


def build_mention_chunks(clusters, starts: list[int]) -> list[list[int]]:
    """Map each mention's character start to a chunk ID."""
    return [
        [mention_to_chunk_id(m[0], starts) for m in cluster]
        for cluster in clusters
    ]


def compute_metrics(mention_chunks, n_chunks):
    return {
        'cluster_break_rate':   cluster_break_rate(mention_chunks),
        'edge_cut_rate':        edge_cut_rate(mention_chunks),
        'entity_concentration': entity_concentration(mention_chunks, n_chunks),
    }

## Load Article List

Article titles come from the parquet (used for baseline boundary approximation).
Cluster files must already exist in `clusters/squad/` — run `generate_squad_clusters.py` first.

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
article_map = dict(zip(df['title'], df['text']))  # title -> full text
titles = sorted(article_map.keys())

# Only process articles that have a cluster file
titles_with_clusters = [t for t in titles if (CLUSTERS_DIR / f'{slugify(t)}.json').exists()]
print(f'Total articles in parquet : {len(titles)}')
print(f'Articles with cluster files: {len(titles_with_clusters)}')

## Compute Metrics per Strategy

In [ ]:
rows = []

for strategy in STRATEGIES:
    for title in titles_with_clusters:
        clusters = load_clusters(title)
        if not clusters:
            continue

        chunks = load_chunks(strategy, title)
        if not chunks:
            continue

        if strategy == 'baseline_258_tok':
            article_len = len(article_map[title])
            starts = chunk_starts_baseline(article_len, len(chunks))
        else:
            starts = chunk_starts_nonbaseline(chunks)

        mention_chunks = build_mention_chunks(clusters, starts)
        m = compute_metrics(mention_chunks, len(chunks))
        m['strategy'] = strategy
        m['title']    = title
        m['n_chunks'] = len(chunks)
        m['n_clusters'] = len(clusters)
        rows.append(m)

results_df = pd.DataFrame(rows)
print(f'Computed metrics for {len(results_df)} (strategy, article) pairs.')
results_df.head()

## Summary Table — Aggregate by Strategy

In [ ]:
agg = (
    results_df
    .groupby('strategy')[['cluster_break_rate', 'edge_cut_rate', 'entity_concentration']]
    .mean()
    .rename(index=STRATEGY_SHORT)
    .sort_values('edge_cut_rate')
)
display(agg.round(4))

## Bar Charts

In [ ]:
metrics_info = [
    ('cluster_break_rate',   'Cluster Break Rate',   '← lower is better'),
    ('edge_cut_rate',        'Edge Cut Rate',         '← lower is better'),
    ('entity_concentration', 'Entity Concentration',  '→ higher is better'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Coreference Metrics by Chunking Strategy — SQuAD', fontweight='bold')

strat_order = list(STRATEGY_SHORT.keys())
labels = [STRATEGY_SHORT[s] for s in strat_order]

for ax, (metric, title, note) in zip(axes, metrics_info):
    vals = [results_df[results_df['strategy'] == s][metric].mean() for s in strat_order]
    bars = ax.bar(labels, vals, color=COLORS, edgecolor='black')
    ax.set_title(f'{title}\n{note}', fontsize=11)
    ax.set_ylabel(metric)
    mean_val = sum(vals) / len(vals)
    ax.axhline(mean_val, color='black', linestyle='--', linewidth=1.2,
               label=f'mean={mean_val:.4f}')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Per-Article Distributions (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Per-Article Metric Distributions by Strategy', fontweight='bold')

for ax, (metric, title, note) in zip(axes, metrics_info):
    data = [results_df[results_df['strategy'] == s][metric].values for s in strat_order]
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f'{title}\n{note}', fontsize=11)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()